# 연구비 판정관 — Colab GPU 실측
같은 벤치(`bench/run_bench.py`)를 Colab에서 돌린다. 런타임 → GPU(T4/L4/A100) 선택 후 위에서부터 실행. 결과 JSON을 내려받아 `bench/results/`에 넣으면 `/bench` 화면에 뜬다.

In [ ]:
!nvidia-smi -L
!pip -q install llama-cpp-python httpx
!curl -L -o qwen2.5-1.5b-instruct-q4_k_m.gguf https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct-GGUF/resolve/main/qwen2.5-1.5b-instruct-q4_k_m.gguf
# llama.cpp 서버 바이너리(CUDA) — 릴리스 zip 사용. 버전은 https://github.com/ggml-org/llama.cpp/releases 에서 최신으로 바꾼다.
!curl -L -o llama.zip https://github.com/ggml-org/llama.cpp/releases/latest/download/llama-b9999-bin-ubuntu-cuda-x64.zip || echo '릴리스 이름 확인 필요'
!unzip -q -o llama.zip -d llama || true
!ls llama/build/bin | head

In [ ]:
import subprocess, time, urllib.request
srv = subprocess.Popen(['llama/build/bin/llama-server','-m','qwen2.5-1.5b-instruct-q4_k_m.gguf','--port','8097','-c','2048','-ngl','99'])
for _ in range(60):
    try:
        urllib.request.urlopen('http://127.0.0.1:8097/health'); break
    except Exception: time.sleep(1)
print('server up')

In [ ]:
# mvp 폴더(bench/·panjeong/extract/·data/answer_key.json)를 zip으로 올린 뒤:
!unzip -q -o mvp-bench.zip -d mvp
%cd mvp
!python bench/run_bench.py --host Colab-$(nvidia-smi --query-gpu=name --format=csv,noheader | head -1 | tr ' ' '-')
!ls bench/results